In [125]:
import os
import pandas as pd
from openai import OpenAI

import numpy as np
import json

In [126]:
from dotenv import load_dotenv
load_dotenv()

# Place API_KEY in the .env file
api_key = os.environ.get('API_KEY')
client = OpenAI(api_key=api_key)

In [127]:
repo_dir = "/Users/haya1/Documents/LanguageModel_Labels/congressional_bills"
os.chdir(repo_dir)


## Generate prompts

In [128]:
llm_dir = os.path.join(repo_dir, "02_llm")
base_prompt = open(os.path.join(llm_dir, "base_prompt_json.txt"), 'r').read()
print("Loaded base question")
# print(base_prompt)

Loaded base question


In [129]:
prompting_strategies = pd.DataFrame(data={
    "Name": [
        "No Modification", 
        "Persona Modification", "Persona Modification", "Persona Modification", "Persona Modification", 
        "Chain-of-Thoughts Prompting", "Chain-of-Thoughts Prompting", "Chain-of-Thoughts Prompting", 
        # "Few-Shot Prompting", "Few-Shot Prompting", "Few-Shot Prompting"
        ],
    "BeforeQuestion": [
        "",
        "You are a knowledgeable political analyst. ",
        "Answer this question as if you are a political scientist that studies legislation in the United States Congress. ",
        "Answer this question as if you are an expert in United States politics. ",
        "Answer this question as if you were a helpful research assistant for a political scientist. ",
        "", "", "",
        # "", "", "",
        ],
    "BeforeAnswer": [
        "", 
        "", "", "", "", 
        "Think carefully. ", 
        "Let's think step by step. Lay out each step. ", 
        "Please provide an explanation for your answer. ",
        # "", "", "",
    ],
    "Explanation":[
        "", 
        "", "", "", "",
        ',\n    "explanation": a one-sentence explanation of your bill category answer',
        ',\n    "explanation": a one-sentence explanation of your bill category answer',
        ',\n    "explanation": a one-sentence explanation of your bill category answer',
        # "", "", "",
    ]
})

In [130]:
data_dir = os.path.join(repo_dir, "01_cleaning_and_data")
bills = pd.read_csv(os.path.join(data_dir, "bills.csv")).groupby('Major').sample(n=2) # TODO: remove .groupby('Major').sample(n=1)
bills = bills[["BillID", "Description"]] 
print(f"Loaded bills data, n = {len(bills)}")

# # Bills used for few-shot prompting
# bills_prompts = bills.groupby('Major').sample(n=3)
# bills_prompts.to_csv(os.path.join(data_dir, "bills_prompts.csv"), index=False)

# # Exclude bills used for prompting from bills data
# condition = bills["BillID"].apply(lambda x: x not in bills_prompts["BillID"])
# bills_no_prompts = bills[condition]
# bills_no_prompts.to_csv(os.path.join(data_dir, "bills_no_prompts.csv"), index=False)

bill_ids = []
strategies = []
prompts = []

for _, bill in bills.iterrows():
    for _, strategy in prompting_strategies.iterrows():
        bill_ids.append(bill["BillID"])
        strategies.append(strategy["Name"])
        prompts.append(base_prompt % (strategy["BeforeQuestion"], bill["Description"], strategy["BeforeAnswer"], strategy["Explanation"]))

prompts = pd.DataFrame(data = {
    "PromptID": list(range(1,len(prompts)+1)),
    "BillID": bill_ids,
    "PromptingStrategy": strategies,
    "Prompt": prompts
})
prompts.to_csv(os.path.join(llm_dir, "prompts.csv"), index=False)

Loaded bills data, n = 42


## Generate responses

In [131]:
# SYSTEM_PROMPT = ""

# define a response function that gives us the LLM's response to a user prompt
def query_llm(prompt, model="gpt-3.5-turbo", temperature=0, num_responses=1):
    response_text = client.chat.completions.create(
        model = model,
        logprobs = True,
        n = num_responses,
        temperature = temperature,
        response_format={"type": "json_object"},
        messages=[
            # {"role": "system",  #  role (either "system", "user", or "assistant") The system message helps set the behavior of the assistant. For example, you can modify the personality of the assistant or provide specific instructions about how it should behave throughout the conversation. However note that the system message is optional and the model’s behavior without a system message is likely to be similar to using a generic message such as "You are a helpful assistant."
            #  "content": system},
            {"role": "user", 
            "content": prompt},
            # {"role": "assistant",
            #  "content": examples} # Assistant messages store previous assistant responses, but can also be written by you to give examples of desired behavior.
        ]
    ).choices[0].message.content

    response_json = json.loads(response_text)
    
    out = [np.nan, np.nan, np.nan] # category, confidence, explanation

    if "category" in response_json.keys():
        out[0] = response_json["category"]

    if "confidence" in response_json.keys():
        out[1] = response_json["confidence"]

    if "explanation" in response_json.keys():
        out[2] = response_json["explanation"]
    
    return out

In [132]:
prompts = pd.read_csv(os.path.join(llm_dir, "prompts.csv"))
print(f"Loaded prompts data, n = {len(prompts)}")

prompt_id = []
bill_id = []
major_llm = []
confidence_llm = []
explanation_llm = []

for i in range(len(prompts)):
    prompt_id.append(prompts["PromptID"][i])
    bill_id.append(prompts["BillID"][i])
    
    response = query_llm(prompt=prompts["Prompt"][i], 
                         model="gpt-3.5-turbo", 
                         temperature=0)
    
    major_llm.append(response[0])
    confidence_llm.append(response[1])
    explanation_llm.append(response[2])

responses = pd.DataFrame(data = {
    "PromptID": prompt_id,
    "MajorLLM": major_llm,
    "ConfidenceLLM": confidence_llm,
    "ExplanationLLM": explanation_llm
})

Loaded prompts data, n = 336


In [133]:
def get_major_text(major_id):
    # note that this is based on our recoding of major topics excluding topic 99
    major_text = {
        1 : "Macroeconomics",
        2 : "Civil Rights, Minority Issues, and Civil Liberties",
        3 : "Health",
        4 : "Agriculture",
        5 : "Labor and Employment",
        6 : "Education",
        7 : "Environment",
        8 : "Energy",
        9 : "Immigration",
        10: "Transportation",
        11: "Law, Crime, and Family Issues",
        12: "Social Welfare",
        13: "Community Development and Housing Issues",
        14: "Banking, Finance, and Domestic Commerce",
        15: "Defense",
        16: "Space, Science, Technology, and Communications",
        17: "Foreign Trade",
        18: "International Affairs and Foreign Aid",
        19: "Government Operations",
        20: "Public Lands and Water Management",
        21: "Arts and Entertainment"
    }
    if pd.notnull(major_id):
        return major_text[major_id]
    else:
        return np.nan
    
responses["MajorTextLLM"] = responses["MajorLLM"].apply(get_major_text)
responses = responses[["PromptID", "MajorLLM", "MajorTextLLM", "ConfidenceLLM", "ExplanationLLM"]]

print(responses.head())

responses.to_csv(os.path.join(llm_dir, "responses.csv"), index=False)

   PromptID  MajorLLM           MajorTextLLM  ConfidenceLLM ExplanationLLM
0         1        19  Government Operations           0.75            NaN
1         2        19  Government Operations           0.85            NaN
2         3        19  Government Operations           0.85            NaN
3         4        19  Government Operations           0.85            NaN
4         5        19  Government Operations           0.85            NaN


In [134]:
bills = pd.read_csv(os.path.join(data_dir, "bills.csv"))[['BillID', 'Major', 'MajorText', 'Description']]
prompts = pd.read_csv(os.path.join(llm_dir, "prompts.csv"))
responses = pd.read_csv(os.path.join(llm_dir, "responses.csv"))

condition = bills["BillID"].apply(lambda x: x in prompts["BillID"].unique())
bills = bills[condition]

df = prompts.merge(responses,  on="PromptID", validate="one_to_one")
df = df.merge(bills, on="BillID", validate="many_to_one")
df = df[["PromptID", "BillID", "PromptingStrategy", "Prompt", "Description", "Major", "MajorText", "MajorLLM", "MajorTextLLM", "ConfidenceLLM", "ExplanationLLM"]]
df.head()

,PromptID,BillID,PromptingStrategy,Prompt,Description,Major,MajorText,MajorLLM,MajorTextLLM,ConfidenceLLM,ExplanationLLM
0,1,91-HR-14366,No Modification,Here is a description of a bill introduced in ...,To provide that the fiscal year of the United ...,1,Macroeconomics,19,Government Operations,0.75,NaN
1,2,91-HR-14366,Persona Modification,You are a knowledgeable political analyst. Her...,To provide that the fiscal year of the United ...,1,Macroeconomics,19,Government Operations,0.85,NaN
2,3,91-HR-14366,Persona Modification,Answer this question as if you are a political...,To provide that the fiscal year of the United ...,1,Macroeconomics,19,Government Operations,0.85,NaN
3,4,91-HR-14366,Persona Modification,Answer this question as if you are an expert i...,To provide that the fiscal year of the United ...,1,Macroeconomics,19,Government Operations,0.85,NaN
4,5,91-HR-14366,Persona Modification,Answer this question as if you were a helpful ...,To provide that the fiscal year of the United ...,1,Macroeconomics,19,Government Operations,0.85,NaN


In [135]:
df.to_csv(os.path.join(llm_dir, "prompts_and_responses.csv"), index=False)